In [9]:
import pandas as pd
import numpy as np
import glob

In [2]:
import geopandas as gpd

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [4]:
RAW_DATA_PATH = "data/raw"

In [5]:
import os
os.listdir(RAW_DATA_PATH)

['data-2023-01.csv',
 'data-2023-02.csv',
 'data-2023-03.csv',
 'data-2023-04.csv',
 'data-2023-05.csv',
 'data-2023-06.csv',
 'data-2023-07.csv',
 'data-2023-08.csv',
 'data-2023-09.csv',
 'data-2023-10.csv',
 'data-2023-11.csv',
 'data-2023-12.csv',
 'Dynamische lijst 2025.xlsx',
 'OPENDATA_MAP_2017-2024.xlsx',
 'richtingen.csv',
 'sites.csv']

In [6]:
# LOAD SENSOR METADATA

site_columns = [
    "sensor_id",
    "site_nr",
    "longitude",
    "latitude",
    "name",
    "domain",
    "road_number",
    "district",
    "municipality",
    "interval",
    "installation_date"
]

sites = pd.read_csv(
    RAW_DATA_PATH + "/sites.csv",
    header=None,
    names=site_columns
)

sites.head()

,sensor_id,site_nr,longitude,latitude,name,domain,road_number,district,municipality,interval,installation_date
0,1,100046096,4.456122,50.916183,Machelen,Vlaamse Overheid A. Wegen enVerkeer,T2110002,AWV212,Machelen,15,2019-08-22
1,2,100052862,4.471690,51.275120,Brasschaat 2,Vlaamse Overheid A. Wegen enVerkeer,N0010002,AWV123,Brasschaat,15,2019-08-22
2,3,100052863,4.472220,51.275030,Brasschaat 1,Vlaamse Overheid A. Wegen enVerkeer,N0010001,AWV123,Brasschaat,15,2019-08-22
3,4,100052864,5.190110,51.160230,Balen 1,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,2019-08-22
4,5,100052865,5.190030,51.160180,Balen 2,Vlaamse Overheid A. Wegen enVerkeer,N0180002,AWV114,Balen,15,2019-08-22


In [7]:
#  LOAD DIRECTIONS

direction_columns = [
    "sensor_id",
    "direction",
    "direction_name"]

directions = pd.read_csv(
    RAW_DATA_PATH + "/richtingen.csv",
    header=None,
    names=direction_columns)

directions.head()

,sensor_id,direction,direction_name
0,1,IN,Machelen Cyclists rich. Brucargo
1,1,OUT,Machelen Cyclists richting Machelen
2,2,IN,Brasschaat 2 Fietsers rich Merksem
3,2,OUT,Brasschaat 2 Fietsers rich Brasschaat
4,3,IN,Brasschaat 1 Fietsers rich Merksem


In [10]:
# LOAD AND AGGREGATE TRAFFIC DATA

traffic_columns = [
    "sensor_id",
    "direction",
    "vehicle_type",
    "start_time",
    "end_time",
    "count"
]

traffic_files = sorted(glob.glob(RAW_DATA_PATH + "/data-2023-*.csv"))

daily_parts = []

for file in traffic_files:
    temp = pd.read_csv(
        file,
        header=None,
        names=traffic_columns)
    
    temp["start_time"] = pd.to_datetime(temp["start_time"])
    temp["date"] = temp["start_time"].dt.date
    temp = temp.dropna(subset=["count"])
    
    daily_temp = temp.groupby(
        ["sensor_id", "date"]
    )["count"].sum().reset_index()
    
    daily_parts.append(daily_temp)

sensor_daily = pd.concat(daily_parts, ignore_index=True)

sensor_daily.head()

,sensor_id,date,count
0,1,2023-01-01,124.0
1,1,2023-01-02,166.0
2,1,2023-01-03,354.0
3,1,2023-01-04,213.0
4,1,2023-01-05,286.0


In [11]:
# MERGE SENSOR METADATA
sensor_daily = sensor_daily.merge(
    sites[[
        "sensor_id",
        "municipality",
        "longitude",
        "latitude",
        "name"]],
    on="sensor_id",
    how="left"
)

sensor_daily.head()

,sensor_id,date,count,municipality,longitude,latitude,name
0,1,2023-01-01,124.0,Machelen,4.456122,50.916183,Machelen
1,1,2023-01-02,166.0,Machelen,4.456122,50.916183,Machelen
2,1,2023-01-03,354.0,Machelen,4.456122,50.916183,Machelen
3,1,2023-01-04,213.0,Machelen,4.456122,50.916183,Machelen
4,1,2023-01-05,286.0,Machelen,4.456122,50.916183,Machelen


In [12]:
sensor_daily.isnull().sum()

sensor_id       0
date            0
count           0
municipality    0
longitude       0
latitude        0
name            0
dtype: int64

In [13]:
sensor_daily.shape

(49596, 7)

In [14]:
# MUNICIPALITY LOCATIONS FOR WEATHER API

municipality_locations = sensor_daily.groupby("municipality")[[
    "latitude",
    "longitude"]].mean().reset_index()

municipality_locations.head()

,municipality,latitude,longitude
0,Aalst,50.934557,4.015630
1,Aalter,51.105837,3.435059
2,Aarschot,50.990840,4.812590
3,Ardooie,50.967835,3.184450
4,As,51.005090,5.603890


In [ ]:
# 8. GET WEATHER DATA FROM OPEN-METEO
import requests

url = "https://archive-api.open-meteo.com/v1/archive"
all_weather = []

for index, row in municipality_locations.iterrows():

    params = {
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "start_date": "2023-01-01",
        "end_date": "2023-12-31",
        "daily": "temperature_2m_mean,precipitation_sum,windspeed_10m_max",
        "timezone": "Europe/Brussels"}

    response = requests.get(url, params=params)
    weather_json = response.json()

    weather_df = pd.DataFrame(weather_json["daily"])
    weather_df["municipality"] = row["municipality"]
    all_weather.append(weather_df)

weather_2023 = pd.concat(all_weather, ignore_index=True)
weather_2023.head()

,time,temperature_2m_mean,precipitation_sum,windspeed_10m_max,municipality
0,2023-01-01,13.2,4.0,33.6,Aalst
1,2023-01-02,9.9,7.0,28.2,Aalst
2,2023-01-03,6.6,0.4,32.1,Aalst
3,2023-01-04,11.2,4.4,33.9,Aalst
4,2023-01-05,10.3,1.2,24.4,Aalst


In [16]:
weather_2023.shape

(25185, 5)

In [17]:
weather_2023["municipality"].nunique()

69

In [18]:
#merge data with sensor_daily
weather_2023["date"] = pd.to_datetime(weather_2023["time"]).dt.date
sensor_weather_daily = sensor_daily.merge(
    weather_2023,
    on=["municipality", "date"],
    how="left")
sensor_weather_daily.head()

,sensor_id,date,count,municipality,longitude,latitude,name,time,temperature_2m_mean,precipitation_sum,windspeed_10m_max
0,1,2023-01-01,124.0,Machelen,4.456122,50.916183,Machelen,2023-01-01,13.2,4.0,34.3
1,1,2023-01-02,166.0,Machelen,4.456122,50.916183,Machelen,2023-01-02,10.2,5.0,30.6
2,1,2023-01-03,354.0,Machelen,4.456122,50.916183,Machelen,2023-01-03,6.7,0.4,31.1
3,1,2023-01-04,213.0,Machelen,4.456122,50.916183,Machelen,2023-01-04,11.1,3.6,36.4
4,1,2023-01-05,286.0,Machelen,4.456122,50.916183,Machelen,2023-01-05,10.3,0.7,25.4


In [19]:
sensor_weather_daily[[
    "temperature_2m_mean",
    "precipitation_sum",
    "windspeed_10m_max"
]].isnull().sum()

temperature_2m_mean    0
precipitation_sum      0
windspeed_10m_max      0
dtype: int64

In [20]:
#build weather features
sensor_weather_daily["is_rainy_day"] = (
    sensor_weather_daily["precipitation_sum"] > 0)

sensor_weather_daily["is_windy_day"] = (
    sensor_weather_daily["windspeed_10m_max"] > 30)
sensor_weather_daily.head()

,sensor_id,date,count,municipality,longitude,latitude,name,time,temperature_2m_mean,precipitation_sum,windspeed_10m_max,is_rainy_day,is_windy_day
0,1,2023-01-01,124.0,Machelen,4.456122,50.916183,Machelen,2023-01-01,13.2,4.0,34.3,True,True
1,1,2023-01-02,166.0,Machelen,4.456122,50.916183,Machelen,2023-01-02,10.2,5.0,30.6,True,True
2,1,2023-01-03,354.0,Machelen,4.456122,50.916183,Machelen,2023-01-03,6.7,0.4,31.1,True,True
3,1,2023-01-04,213.0,Machelen,4.456122,50.916183,Machelen,2023-01-04,11.1,3.6,36.4,True,True
4,1,2023-01-05,286.0,Machelen,4.456122,50.916183,Machelen,2023-01-05,10.3,0.7,25.4,True,False


In [21]:
# SENSOR-LEVEL FEATURES
sensor_features = sensor_weather_daily.groupby("sensor_id").agg({
    "count": ["sum", "mean"],
    "temperature_2m_mean": "mean",
    "precipitation_sum": "mean",
    "windspeed_10m_max": "mean",
    "is_rainy_day": "mean",
    "is_windy_day": "mean"})

sensor_features.columns = [
    "total_traffic",
    "avg_daily_traffic",
    "avg_temperature",
    "avg_precipitation",
    "avg_wind_speed",
    "rainy_day_ratio",
    "windy_day_ratio"]

sensor_features = sensor_features.reset_index()
sensor_features.head()

,sensor_id,total_traffic,avg_daily_traffic,avg_temperature,avg_precipitation,avg_wind_speed,rainy_day_ratio,windy_day_ratio
0,1,143588.0,393.391781,11.977534,2.678356,22.038082,0.679452,0.205479
1,2,290861.0,796.879452,11.869589,3.100274,21.469315,0.704110,0.167123
2,3,259489.0,710.928767,11.869589,3.100274,21.469315,0.704110,0.167123
3,4,62706.0,171.797260,11.917260,2.897808,20.843562,0.687671,0.161644
4,5,67837.0,185.854795,11.917260,2.897808,20.843562,0.687671,0.161644


In [22]:
"joined" in globals(),
"accident_counts" in globals()

False

In [23]:
# LOAD AND PREPROCESS ACCIDENT DATA

accidents = pd.read_excel(
    RAW_DATA_PATH + "/OPENDATA_MAP_2017-2024.xlsx")

flanders_accidents = accidents[
    accidents["TX_RGN_COLLISION_NL"] == "Vlaams Gewest"]

bike_accidents = flanders_accidents[
    (flanders_accidents["TX_ROAD_USR_TYPE1_NL"] == "Fiets") |
    (flanders_accidents["TX_ROAD_USR_TYPE2_NL"] == "Fiets")]

bike_accidents_geo = bike_accidents.dropna(
    subset=["MS_X_COORD", "MS_Y_COORD"])

bike_accidents_geo.shape

(53524, 45)

In [24]:
# ACCIDENT GEODATAFRAME

accidents_gdf = gpd.GeoDataFrame(
    bike_accidents_geo,
    geometry=gpd.points_from_xy(
        bike_accidents_geo["MS_X_COORD"],
        bike_accidents_geo["MS_Y_COORD"]),
    crs="EPSG:31370")
accidents_gdf.head()

,DT_YEAR_COLLISION,DT_MONTH_COLLISION,DT_TIME,CD_NIS,TX_RGN_COLLISION_FR,TX_RGN_COLLISION_NL,TX_PROV_COLLISION_FR,TX_PROV_COLLISION_NL,TX_MUNTY_COLLISION_FR,TX_MUNTY_COLLISION_NL,...,CD_ROAD_USR_TYPE2,TX_ROAD_USR_TYPE2_FR,TX_ROAD_USR_TYPE2_NL,CD_COLLISION_TYPE,TX_COLLISON_TYPE_FR,TX_COLLISION_TYPE_NL,CD_OBSTACLES,TX_OBSTACLES_FR,TX_OBSTACLES_NL,geometry
1,2017,1,0,11002,Région flamande,Vlaams Gewest,Province d’Anvers,Provincie Antwerpen,Anvers,Antwerpen,...,9,Voiture,personenauto,2,Entre 2 conducteurs: Collision frontale,Tussen 2 bestuurders: Frontale botsing,0,Pas d’obstacle,Geen hindernis,POINT (152268.069 209633.121)
13,2017,1,7,11001,Région flamande,Vlaams Gewest,Province d’Anvers,Provincie Antwerpen,Aartselaar,Aartselaar,...,3,Bicyclette,Fiets,4,Entre 2 conducteurs: Par le côté (avant/derriè...,Tss. 2 best.: langs opzij (voor-/achterkant-fl...,0,Pas d’obstacle,Geen hindernis,POINT (150662.902 203118.932)
200,2017,1,7,23094,Région flamande,Vlaams Gewest,Province du Brabant flamand,Provincie Vlaams-Brabant,Zaventem,Zaventem,...,3,Bicyclette,Fiets,4,Entre 2 conducteurs: Par le côté (avant/derriè...,Tss. 2 best.: langs opzij (voor-/achterkant-fl...,0,Pas d’obstacle,Geen hindernis,POINT (157232.033 175893.359)
1221,2017,1,16,11002,Région flamande,Vlaams Gewest,Province d’Anvers,Provincie Antwerpen,Anvers,Antwerpen,...,9,Voiture,personenauto,3,Entre 2 conducteurs: Par l'arrière,Tussen 2 bestuurders: Langs achteren,0,Pas d’obstacle,Geen hindernis,POINT (154445.154 207467.044)
1222,2017,1,15,11002,Région flamande,Vlaams Gewest,Province d’Anvers,Provincie Antwerpen,Anvers,Antwerpen,...,8,Piéton,Voetganger,5,Avec un piéton,Met een voetganger,0,Pas d’obstacle,Geen hindernis,POINT (154899.366 216114.622)


In [25]:
# SENSOR GEODATAFRAME

sensor_features = sensor_features.merge(
    sites[["sensor_id", "longitude", "latitude", "municipality", "name"]],
    on="sensor_id",
    how="left")

sensors_gdf = gpd.GeoDataFrame(
    sensor_features,
    geometry=gpd.points_from_xy(
        sensor_features["longitude"],
        sensor_features["latitude"]
    ),
    crs="EPSG:4326")

sensors_gdf = sensors_gdf.to_crs("EPSG:31370")

sensors_gdf.head()

,sensor_id,total_traffic,avg_daily_traffic,avg_temperature,avg_precipitation,avg_wind_speed,rainy_day_ratio,windy_day_ratio,longitude,latitude,municipality,name,geometry
0,1,143588.0,393.391781,11.977534,2.678356,22.038082,0.679452,0.205479,4.456122,50.916183,Machelen,Machelen,POINT (156143.904 178432.685)
1,2,290861.0,796.879452,11.869589,3.100274,21.469315,0.704110,0.167123,4.471690,51.275120,Brasschaat,Brasschaat 2,POINT (157182.957 218365.646)
2,3,259489.0,710.928767,11.869589,3.100274,21.469315,0.704110,0.167123,4.472220,51.275030,Brasschaat,Brasschaat 1,POINT (157219.957 218355.684)
3,4,62706.0,171.797260,11.917260,2.897808,20.843562,0.687671,0.161644,5.190110,51.160230,Balen,Balen 1,POINT (207457.483 205896.897)
4,5,67837.0,185.854795,11.917260,2.897808,20.843562,0.687671,0.161644,5.190030,51.160180,Balen,Balen 2,POINT (207451.948 205891.273)


In [26]:
# SPATIAL JOIN: ACCIDENTS TO NEAREST SENSOR-500m

joined = gpd.sjoin_nearest(
    accidents_gdf,
    sensors_gdf[["sensor_id", "geometry"]],
    how="left",
    distance_col="distance_to_sensor")

nearby_accidents = joined[
    joined["distance_to_sensor"] <= 500]

nearby_accidents.shape

(1237, 49)

In [27]:
#accident count for each sensor
#RISK DATA

accident_counts = nearby_accidents.groupby(
    "sensor_id").size().reset_index(name="accident_count")

risk_data = sensor_features[[
    "sensor_id",
    "total_traffic"]].merge(
    accident_counts,
    on="sensor_id",
    how="left")

risk_data["accident_count"] = risk_data["accident_count"].fillna(0)

risk_data["risk_rate"] = (
    risk_data["accident_count"] / risk_data["total_traffic"])

risk_data.head()

,sensor_id,total_traffic,accident_count,risk_rate
0,1,143588.0,0.0,0.000000
1,2,290861.0,5.0,0.000017
2,3,259489.0,6.0,0.000023
3,4,62706.0,0.0,0.000000
4,5,67837.0,4.0,0.000059


In [28]:
#add risk data
# FINAL MODEL DATASET

sensor_features = sensor_features.merge(
    risk_data[[
        "sensor_id",
        "accident_count",
        "risk_rate"]],
    on="sensor_id",
    how="left")

sensor_features[[
    "accident_count",
    "risk_rate"]] = sensor_features[[
    "accident_count",
    "risk_rate"]].fillna(0)

sensor_features.head()

,sensor_id,total_traffic,avg_daily_traffic,avg_temperature,avg_precipitation,avg_wind_speed,rainy_day_ratio,windy_day_ratio,longitude,latitude,municipality,name,accident_count,risk_rate
0,1,143588.0,393.391781,11.977534,2.678356,22.038082,0.679452,0.205479,4.456122,50.916183,Machelen,Machelen,0.0,0.000000
1,2,290861.0,796.879452,11.869589,3.100274,21.469315,0.704110,0.167123,4.471690,51.275120,Brasschaat,Brasschaat 2,5.0,0.000017
2,3,259489.0,710.928767,11.869589,3.100274,21.469315,0.704110,0.167123,4.472220,51.275030,Brasschaat,Brasschaat 1,6.0,0.000023
3,4,62706.0,171.797260,11.917260,2.897808,20.843562,0.687671,0.161644,5.190110,51.160230,Balen,Balen 1,0.0,0.000000
4,5,67837.0,185.854795,11.917260,2.897808,20.843562,0.687671,0.161644,5.190030,51.160180,Balen,Balen 2,4.0,0.000059


In [29]:
# TARGET VARIABLE-25% of top high risk sensors
risk_threshold = sensor_features["risk_rate"].quantile(0.75)

sensor_features["high_risk"] = (
    sensor_features["risk_rate"] >= risk_threshold)

sensor_features["high_risk"].value_counts()

high_risk
False    105
True      35
Name: count, dtype: int64

In [30]:
# MACHINE LEARNING MODEL
X = sensor_features[[
    "total_traffic",
    "avg_daily_traffic",
    "avg_temperature",
    "avg_precipitation",
    "avg_wind_speed",
    "rainy_day_ratio",
    "windy_day_ratio"]]

y = sensor_features["high_risk"]
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y)

model = RandomForestClassifier(
    random_state=42,
    class_weight="balanced")

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.7857142857142857
              precision    recall  f1-score   support

       False       0.83      0.90      0.86        21
        True       0.60      0.43      0.50         7

    accuracy                           0.79        28
   macro avg       0.71      0.67      0.68        28
weighted avg       0.77      0.79      0.77        28

